# 01 · Polaris judge calibration

Does the probability-weighted VLM judge rank caption quality closer to humans than CLIPScore,
and does it pass the pre-registered gate for the full pool run? See `PREREGISTRATION.md`.

Inputs (from `scripts/score_polaris.py` and `scripts/score_handrate.py`, pulled from S3):
`data/scores/polaris_{train,test}.parquet`, `polaris_test_thumbs.parquet`, `handrate.parquet` (optional).
Runs on a laptop, no GPU.

In [ ]:
import io, json

from datacurate.io import DATA, RESULTS, load_config

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

from datacurate.calibration import ece, fit_isotonic, mae, reliability_table
from datacurate.polaris import rater_pairs
from datacurate.stats import correlation_table, corr, go_no_go

cfg = load_config("configs/polaris_calibration.yaml")
BOOT = dict(n_boot=cfg["gate"]["n_boot"], seed=cfg["seed"], ci=cfg["gate"]["ci"])
S = DATA / "scores"
train = pd.read_parquet(S / "polaris_train.parquet")
test = pd.read_parquet(S / "polaris_test.parquet")
scoring = json.loads((RESULTS / "polaris_scoring.json").read_text())
print(scoring["judge_model"], "rubric", scoring["rubric_sha"])
print(f"train pairs {len(train):,} | test pairs {len(test):,} | test images {test.img_hash.nunique():,}")

## 1 · Sanity: did the judge actually answer?

In [ ]:
for name, df in [("train", train), ("test", test)]:
    print(f"{name}: parse failures {(df.digit_mass < 0.5).mean():.2%}, "
          f"median digit mass {df.digit_mass.median():.3f}")
print("argmax histogram (test):", test.judge_argmax.value_counts().sort_index().to_dict())
# Failures get the median score so rank metrics stay defined; they are counted above.
for df in (train, test):
    df["judge"] = df.judge.fillna(df.judge.median())

## 2 · Human ceiling

Two different raters of the same (image, caption) pair, compared with each other. The judge is compared
with a single rater on the same pairs, so the comparison is like for like.

In [ ]:
rp = rater_pairs(test, seed=cfg["seed"])
ceiling = correlation_table(
    rp.assign(r2_as_scorer=rp.r2), target="r1", scorers=["r2_as_scorer", "judge", "clipscore"],
    groups="img_hash", **BOOT,
)
print(f"{len(rp):,} pairs with 2+ ratings")
ceiling.round(3)

## 3 · Main result: rank agreement with mean human score (Polaris test)

In [ ]:
SCORERS = ["judge", "judge_argmax", "clipscore"]
DIFFS = [("judge", "clipscore"), ("judge", "judge_argmax")]
spearman = correlation_table(test, "human", SCORERS, "img_hash", diffs=DIFFS, **BOOT)
kendall = correlation_table(test, "human", SCORERS, "img_hash", method="kendall", **BOOT)
display(spearman.round(3).rename_axis("Spearman"))
display(kendall.round(3).rename_axis("Kendall"))

## 4 · Length bias

If a scorer's correlation with caption length exceeds the humans', it is rewarding length.
Spearman within length tertiles shows whether ranking holds once length is controlled.

In [ ]:
test["len_bin"] = pd.qcut(test.n_words, 3, labels=["short", "medium", "long"])
by_len = pd.DataFrame({
    b: {s: corr(g[s], g.human) for s in SCORERS} | {"n": len(g)}
    for b, g in test.groupby("len_bin", observed=True)
}).T
display(by_len.round(3))
length_corr = {s: corr(test[s], test.n_words) for s in ["human", *SCORERS]}
pd.Series(length_corr, name="Spearman with n_words").round(3)

## 5 · Calibration (fit on train, evaluate on test)

Isotonic maps are monotonic, so Spearman does not change. What changes is the scale:
MAE and ECE against the human 1-5 score. CLIPScore is not on a 1-5 scale, so it has no "before".

In [ ]:
iso_j = fit_isotonic(train.judge, train.human)
iso_c = fit_isotonic(train.clipscore, train.human)
test["judge_cal"] = iso_j.predict(test.judge)
test["clip_cal"] = iso_c.predict(test.clipscore)

calib = pd.DataFrame({
    "judge raw": {"MAE": mae(test.judge, test.human), "ECE": ece(test.judge, test.human)},
    "judge isotonic": {"MAE": mae(test.judge_cal, test.human), "ECE": ece(test.judge_cal, test.human)},
    "clipscore isotonic": {"MAE": mae(test.clip_cal, test.human), "ECE": ece(test.clip_cal, test.human)},
}).T
display(calib.round(3))

fig, ax = plt.subplots(figsize=(4.5, 4.5))
for col, label in [("judge", "judge raw"), ("judge_cal", "judge isotonic"), ("clip_cal", "clipscore isotonic")]:
    rt = reliability_table(test[col], test.human)
    ax.plot(rt.mean_pred, rt.mean_human, marker="o", label=label)
ax.plot([1, 5], [1, 5], ls="--", c="grey", lw=1)
ax.set(xlabel="mean predicted score", ylabel="mean human score", xlim=(1, 5), ylim=(1, 5),
       title="Reliability, Polaris test")
ax.legend(frameon=False)
RESULTS.mkdir(exist_ok=True)
fig.savefig(RESULTS / "polaris_reliability.png", dpi=150, bbox_inches="tight")

## 6 · In-domain check: hand-rated CC12M (blind)

In [ ]:
hand_path = S / "handrate.parquet"
hand = None
if hand_path.exists():
    h = pd.read_parquet(hand_path)
    h["judge"] = h.judge.fillna(h.judge.median())
    hand = correlation_table(h, "rating", ["judge", "clipscore"], "key", diffs=[("judge", "clipscore")], **BOOT)
    print(f"{len(h)} hand-rated pairs")
    display(hand.round(3))
else:
    print("Not scored yet: run scripts/score_handrate.py after rating. The gate stays PENDING.")

## 7 · Pre-registered gate

In [ ]:
gate = go_no_go(spearman, hand, "judge", "clipscore", cfg["gate"]["min_spearman_gain"])
gate |= {
    "judge_model": scoring["judge_model"], "rubric_sha": scoring["rubric_sha"],
    "human_ceiling_spearman": float(ceiling.loc["r2_as_scorer", "point"]),
    "calibration": calib.round(4).to_dict(orient="index"),
}
(RESULTS / "polaris_gate.json").write_text(json.dumps(gate, indent=2))
print(json.dumps(gate, indent=2))

## 8 · Disagreements → failure modes

The 20 largest |calibrated judge − human| gaps on test. Fill in `failure_mode`
(e.g. hallucinated object, too short, count error, judge misses small text) in the exported CSV.

In [ ]:
thumbs = pd.read_parquet(S / "polaris_test_thumbs.parquet").set_index("img_hash").thumb
test["gap"] = test.judge_cal - test.human
worst = test.reindex(test.gap.abs().sort_values(ascending=False).index).head(20)

def row_html(r):
    import base64
    src = "data:image/jpeg;base64," + base64.b64encode(thumbs[r.img_hash]).decode()
    side = "judge too high" if r.gap > 0 else "judge too low"
    return (f"<tr><td><img src='{src}' width=160></td><td><b>{r.cand}</b><br>"
            f"human {r.human:.2f} · judge {r.judge_cal:.2f} ({side}) · CLIP {r.clipscore:.2f}<br>"
            f"<small>ref: {r.refs[0]}</small></td></tr>")

display(HTML("<table>" + "".join(row_html(r) for r in worst.itertuples()) + "</table>"))
worst.assign(failure_mode="")[["img_hash", "cand", "human", "judge", "judge_cal", "clipscore", "n_words", "gap", "failure_mode"]].to_csv(
    RESULTS / "polaris_disagreements.csv", index=False)

## 9 · Throughput (from the scoring run)

In [ ]:
smoke = sorted(RESULTS.glob("smoke_*.json"))
rows = [r for p in smoke for r in json.loads(p.read_text()).get("throughput", [])]
display(pd.DataFrame(rows).round(3) if rows else "Run smoke_test_judge.py --benchmark for the variant table.")
pd.DataFrame({s: {"judge img/s": scoring[s]["judge"]["images_per_s"], "judge $/1K": scoring[s]["judge"]["usd_per_1k_images"],
                  "clip img/s": scoring[s]["clip"]["images_per_s"]} for s in ("train", "test") if s in scoring}).T